## Section 8: Neural Network Selection and Training

In this section, we will:
- Select and train a **Neural Network Model** 
- Provide justification for model choices
- Train models and monitor performance on validation set

### Neural Network Model: Feedforward MLP Regressor
**Justification:** A Multi-Layer Perceptron (MLP) is the standard neural network architecture for structured tabular regression. Unlike CNNs (which require spatial structure) or RNNs (which require sequential data), an MLP directly processes the 27 engineered numeric features through fully connected layers. Its hidden layers with ReLU activations can capture the **non-linear relationship** between HP and CR identified in EDA (where HP scales ~10x from CR 5 to CR 20+), as well as **interaction effects** between features (e.g., a Tiny creature with `Is_Legendary=1` breaking the size-CR rule). Dropout and batch normalization provide regularization to prevent overfitting on the 2,183-sample training set.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Load the 70/15/15 neural network splits
X_train = pd.read_csv('../data/final_split/X_train_nn.csv')
X_val = pd.read_csv('../data/final_split/X_val_nn.csv')
X_test = pd.read_csv('../data/final_split/X_test_nn.csv')
y_train = pd.read_csv('../data/final_split/y_train_nn.csv').values.ravel()
y_val = pd.read_csv('../data/final_split/y_val_nn.csv').values.ravel()
y_test = pd.read_csv('../data/final_split/y_test_nn.csv').values.ravel()

print(f"Training set:   {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")
print(f"\nFeatures: {X_train.columns.tolist()}")

In [ ]:
# Convert data to PyTorch tensors
# DataLoaders are NOT created here because batch_size is a hyperparameter
# that will be tuned during grid search
X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
X_val_t = torch.tensor(X_val.values, dtype=torch.float32)
X_test_t = torch.tensor(X_test.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print(f"Tensor shapes:")
print(f"  X_train: {X_train_t.shape}  y_train: {y_train_t.shape}")
print(f"  X_val:   {X_val_t.shape}  y_val:   {y_val_t.shape}")
print(f"  X_test:  {X_test_t.shape}  y_test:  {y_test_t.shape}")

#### MLP Architecture (Parameterized)
The model class accepts variable hyperparameters so the grid search can test different configurations:
- `hidden_layers` — tuple defining neuron count per layer, e.g. `(128, 64, 32)` creates 3 hidden layers
- `dropout` — dropout rate applied uniformly after each hidden layer

Each hidden layer follows the same pattern: **Linear → BatchNorm → ReLU → Dropout**

```
Input(27) → [Dense(n) → BatchNorm → ReLU → Dropout] × num_layers → Output(1)
```

In [ ]:
class CRPredictor(nn.Module):
    """
    Feedforward MLP for predicting D&D monster Challenge Rating.
    
    Args:
        input_dim:     Number of input features (27)
        hidden_layers: Tuple of neuron counts per hidden layer, e.g. (128, 64, 32)
        dropout:       Dropout rate applied after each hidden layer
    """
    
    def __init__(self, input_dim, hidden_layers=(128, 64, 32), dropout=0.3):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for neurons in hidden_layers:
            layers.extend([
                nn.Linear(prev_dim, neurons),
                nn.BatchNorm1d(neurons),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = neurons
        
        # Output layer: single neuron, no activation (regression)
        layers.append(nn.Linear(prev_dim, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Quick test: build a model with default hyperparameters
INPUT_DIM = X_train.shape[1]
test_model = CRPredictor(INPUT_DIM, hidden_layers=(128, 64, 32), dropout=0.3)
print(test_model)

total_params = sum(p.numel() for p in test_model.parameters())
print(f"\nTotal parameters: {total_params:,}")

#### Hyperparameter Tuning: Grid Search
We systematically test every combination of four key hyperparameters to find the best configuration. Each combination trains a fresh model using early stopping on the validation set. The combination with the **lowest validation loss** wins.

**Search Space:**
| Hyperparameter | Values Tested | Why These |
|---|---|---|
| Learning Rate | 0.01, 0.001, 0.0001 | Covers aggressive → conservative step sizes |
| Batch Size | 32, 64, 128 | Trades gradient noise vs. stability |
| Hidden Layers | (128,64,32), (64,32,16), (256,128,64) | Small → medium → large capacity |
| Dropout | 0.2, 0.3 | Light vs. moderate regularization |

**Total combinations:** 3 × 3 × 3 × 2 = **54 trials**

In [ ]:
from itertools import product
import time

# ── Search space ──
param_grid = {
    'learning_rate': [0.01, 0.001, 0.0001],
    'batch_size':    [32, 64, 128],
    'hidden_layers': [(128, 64, 32), (64, 32, 16), (256, 128, 64)],
    'dropout':       [0.2, 0.3],
}

# Grid search settings (reduced epochs for speed — just enough to compare)
GS_EPOCHS = 100
GS_PATIENCE = 15

def run_trial(learning_rate, batch_size, hidden_layers, dropout):
    """Train one model configuration and return its best validation loss."""
    # Reset seed so every trial starts from the same initialization
    torch.manual_seed(42)
    
    # Build model with this configuration
    model = CRPredictor(INPUT_DIM, hidden_layers=hidden_layers, dropout=dropout)
    
    # Build DataLoaders with this batch size
    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(X_val_t, y_val_t),
        batch_size=batch_size, shuffle=False
    )
    
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    
    # Train with early stopping
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(1, GS_EPOCHS + 1):
        # Train
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        
        # Validate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                val_loss += criterion(model(X_batch), y_batch).item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= GS_PATIENCE:
                break
    
    return best_val_loss

# ── Run grid search ──
keys = list(param_grid.keys())
combos = list(product(*param_grid.values()))
results = []

print(f"Grid Search: {len(combos)} combinations × up to {GS_EPOCHS} epochs each")
print("=" * 80)

start_time = time.time()
for i, combo in enumerate(combos, 1):
    params = dict(zip(keys, combo))
    val_loss = run_trial(**params)
    results.append({**params, 'val_loss': val_loss})
    
    if i % 9 == 0 or i == 1 or i == len(combos):
        elapsed = time.time() - start_time
        print(f"  [{i:2d}/{len(combos)}] lr={params['learning_rate']}, bs={params['batch_size']}, "
              f"layers={params['hidden_layers']}, drop={params['dropout']} "
              f"→ val_loss={val_loss:.4f}  ({elapsed:.0f}s elapsed)")

total_time = time.time() - start_time
print(f"\nGrid search complete in {total_time:.0f}s")

In [ ]:
# ── Grid Search Results ──
results_df = pd.DataFrame(results).sort_values('val_loss')

print("=== TOP 10 HYPERPARAMETER COMBINATIONS ===")
display(results_df.head(10).reset_index(drop=True))

# Extract the best configuration
best = results_df.iloc[0]
BEST_LR = best['learning_rate']
BEST_BS = int(best['batch_size'])
BEST_LAYERS = best['hidden_layers']
BEST_DROPOUT = best['dropout']

print(f"\n{'='*50}")
print(f"BEST CONFIGURATION (val_loss = {best['val_loss']:.4f}):")
print(f"  Learning Rate:  {BEST_LR}")
print(f"  Batch Size:     {BEST_BS}")
print(f"  Hidden Layers:  {BEST_LAYERS}")
print(f"  Dropout:        {BEST_DROPOUT}")
print(f"{'='*50}")

#### Final Training with Best Hyperparameters
Now we retrain the model from scratch using the winning hyperparameters from grid search — but this time with the full training budget (300 epochs, patience=25, learning rate scheduler). The grid search used shorter runs (100 epochs, patience=15) just to compare configurations quickly; the final training gives the best architecture enough time to fully converge.

In [ ]:
# Final Training with best hyperparameters

results_df = pd.DataFrame(results).sort_values('val_loss')
best = results_df.iloc[0]
best_lr = float(best['learning_rate'])
best_bs = int(best['batch_size'].item() if hasattr(best['batch_size'], 'item') else best['batch_size'])
best_layers = best['hidden_layers']
best_dropout = float(best['dropout'])

torch.manual_seed(42)

epochs = 300
patience = 25

model = CRPredictor(INPUT_DIM, hidden_layers=best_layers, dropout=best_dropout)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=best_bs, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(X_val_t, y_val_t),
    batch_size=best_bs, shuffle=False
)

criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=best_lr, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10
)

print(f"Final model config:")
print(f"  Architecture:   {best_layers}")
print(f"  Dropout:        {best_dropout}")
print(f"  Learning Rate:  {best_lr}")
print(f"  Batch Size:     {best_bs}")
print(f"  Max Epochs:     {epochs} (early stopping patience={patience})")
print(f"  Parameters:     {sum(p.numel() for p in model.parameters()):,}")

# ── Training loop with early stopping ──
train_losses = []
val_losses = []
best_val_loss = float('inf')
best_model_state = None
epochs_without_improvement = 0

for epoch in range(1, epochs + 1):
    # Train
    model.train()
    total_train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * X_batch.size(0)
    train_loss = total_train_loss / len(train_loader.dataset)
    
    # Validate
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            total_val_loss += criterion(model(X_batch), y_batch).item() * X_batch.size(0)
    val_loss = total_val_loss / len(val_loader.dataset)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    
    if epoch % 25 == 0 or epoch == 1:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {current_lr:.6f}")
    
    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch}. Best val loss: {best_val_loss:.4f}")
        break

# Restore best weights
model.load_state_dict(best_model_state)
print(f"\nTraining complete. Best validation loss: {best_val_loss:.4f}")

In [ ]:
# Plot training vs validation loss curves
plt.style.use('fivethirtyeight')
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(train_losses, label='Train Loss', color='#30a2da', linewidth=2)
ax.plot(val_losses, label='Validation Loss', color='#fc4f30', linewidth=2)
ax.axvline(x=len(train_losses) - PATIENCE, color='gray', linestyle='--', alpha=0.5, label='Early Stop Trigger')

ax.set_title('MLP Training Progress: Loss Over Epochs')
ax.set_xlabel('Epoch')
ax.set_ylabel('Huber Loss')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Best val loss:    {best_val_loss:.4f}")
print(f"Total epochs run: {len(train_losses)}")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Evaluate on validation set
model.eval()
with torch.no_grad():
    y_val_pred = model(X_val_t).numpy().ravel()

val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
val_r2 = r2_score(y_val, y_val_pred)

print("=== MLP VALIDATION SET RESULTS ===")
print(f"  MAE:  {val_mae:.4f} (average error of ~{val_mae:.1f} CR)")
print(f"  RMSE: {val_rmse:.4f}")
print(f"  R²:   {val_r2:.4f}")

# Scatter plot: Predicted vs Actual CR (Validation)
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_val, y_val_pred, alpha=0.5, color='teal', edgecolors='black', linewidth=0.3, s=40)
ax.plot([0, 30], [0, 30], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_title(f'MLP Validation: Predicted vs Actual CR\nR² = {val_r2:.4f} | MAE = {val_mae:.2f} | RMSE = {val_rmse:.2f}')
ax.set_xlabel('Actual CR')
ax.set_ylabel('Predicted CR')
ax.set_xlim(-1, 32)
ax.set_ylim(-1, 32)
ax.legend()
plt.tight_layout()
plt.show()

#### Final Evaluation on Test Set
We evaluate the trained model on the held-out test set (468 monsters) using three metrics:
- **MAE** — average CR prediction error in human-readable units
- **RMSE** — penalizes large errors on rare high-CR bosses
- **R²** — proportion of CR variance explained by the model (1.0 = perfect)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Generate predictions on the test set
model.eval()
with torch.no_grad():
    y_pred_t = model(X_test_t)
    y_pred = y_pred_t.numpy().ravel()

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== MLP TEST SET RESULTS ===")
print(f"  MAE:  {mae:.4f} (average error of ~{mae:.1f} CR)")
print(f"  RMSE: {rmse:.4f}")
print(f"  R²:   {r2:.4f}")

# Scatter plot: Predicted vs Actual CR
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(y_test, y_pred, alpha=0.5, color='teal', edgecolors='black', linewidth=0.3, s=40)
ax.plot([0, 30], [0, 30], 'r--', linewidth=2, label='Perfect Prediction')

ax.set_title(f'MLP: Predicted vs Actual Challenge Rating\nR² = {r2:.4f} | MAE = {mae:.2f} | RMSE = {rmse:.2f}')
ax.set_xlabel('Actual CR')
ax.set_ylabel('Predicted CR')
ax.set_xlim(-1, 32)
ax.set_ylim(-1, 32)
ax.legend()
plt.tight_layout()
plt.show()

# Show sample predictions
sample_df = pd.DataFrame({
    'Actual_CR': y_test[:15],
    'Predicted_CR': np.round(y_pred[:15], 2),
    'Error': np.round(np.abs(y_test[:15] - y_pred[:15]), 2)
})
print("\n--- Sample Predictions (First 15) ---")
display(sample_df)

### Error Analysis and Model Tuning

In this section, we will:
- Analyze errors made by the model
- Identify patterns in prediction errors 
- Perform hyperparameter tuning
- Improve model performance through adjustments
